<h1 style="text-align: center; font-weight: bold;">
Évaluation intermédiaire - Python pour la data science
</h1>



## **groupe : 4A**
**Amira BARHOUMI**  
**Yassine MELLOUL**  
**Antoine FOUCART**



<div style="text-align: right;font-weight: bold;">
Chargé de TD : Julien PRAMIL
</div>


# **Importation des données :**

In [ ]:
import pandas as pd

df = pd.read_csv(
    'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
)
df.head()

 ## **1. Explorations générales**
 ### *Question 1 - Création des variables :*

In [ ]:
df["code_commune"] = (
    df["code_departement"].astype(str)
    + df["code_commune"].astype(str).str.zfill(3)
)

# création colonne candidat
df["candidat"] = df["prenom"] + " " + df["nom"]
df.head(5)

In [ ]:
#test : 
df[df["libelle_commune"].str.contains("Montrouge", na=False)][["code_departement", "code_commune"]].head(3)

### *Question 2 - Nombre de candidats*



In [ ]:

df["candidat"].unique()
candidats = df["candidat"].dropna().nunique()
f"En 2022, il y avait {candidats} candidats à l'élection présidentielle."

### *Question 3 - Scores nationaux*

In [ ]:
scores_nationaux = df.groupby("candidat")["voix"].sum().reset_index()
scores_nationaux.columns = ["candidat", "votes"]

total_exprime = scores_nationaux["votes"].sum()

scores_nationaux["score"] = (scores_nationaux["votes"] / total_exprime * 100).round(2)

scores_nationaux.sort_values("votes", ascending=False).reset_index(drop=True)

Mise en forme du tableau : 

In [ ]:
!pip install great_tables

In [ ]:
from great_tables import GT
scores_affichage = scores_nationaux.sort_values("votes", ascending=False).reset_index(drop=True)
scores_affichage["score"] = scores_affichage["score"].apply(lambda x: f"{x:.2f}%")
(
    GT(scores_affichage)
    .tab_header(
        title="Elections présidentielles ",
        subtitle="Résultats du premier tour (📅 10 avril 2022)"
    )
    .cols_label(
        candidat="Candidat",
        votes="Nombre votes (total)",
        score="Score (% votes exprimés)"
    )
)

## **2. Comparaison départements / nationale**
### *Question 4 - Scores par département*

In [ ]:
score_departements = (
    df
    .groupby(["code_departement", "candidat"])["voix"]
    .sum()
    .reset_index()
)

score_departements.columns = ["code_departement", "candidat", "votes"]

# total des votes exprimés par département
total_dep = (
    score_departements
    .groupby("code_departement")["votes"]
    .sum()
    .reset_index(name="total_votes")
)

# merge
score_departements = score_departements.merge(total_dep, on="code_departement")

# score
score_departements["score"] = (
    score_departements["votes"] / score_departements["total_votes"] * 100
).round(2)

# tri 
score_departements = score_departements.sort_values(
    ["code_departement", "votes"],
    ascending=[True, False]
).reset_index(drop=True)

score_departements = score_departements.drop(columns="total_votes")

score_departements[score_departements["code_departement"] == "11"]

### *Question 5 - Comparaison avec le niveau national*

In [ ]:
scores_nationaux = scores_nationaux.rename(columns={
    "votes": "votes_national",
    "score": "score_national"
})

score_departements = score_departements.rename(columns={
    "votes": "votes_departement"
})

score_departements = score_departements.merge(
    scores_nationaux,
    on="candidat"
)

score_departements[
    score_departements["code_departement"] == "11"
].sort_values("score", ascending=False)

score_departements[
    score_departements["code_departement"] == "11"
].sort_values("score", ascending=False)

### *Question 6 - Surreprésentation*

**Définition de la surreprésentation :**

La surreprésentation mesure l'écart relatif entre le score d’un candidat dans un département et son score au niveau national.

La formule est la suivante :

$\text{surreprésentation} = \frac{\text{score}_{département} - \text{score}_{national}}{\text{score}_{national}} \times 100$

**Exemple**

Si un candidat obtient :
- 30 % dans un département
- 15 % au niveau national

Alors :

$\frac{30 - 15}{15} \times 100 = 100\%$

Le candidat est donc surreprésenté de 100 % dans ce département (son score est deux fois plus élevé que la moyenne nationale).

In [ ]:
score_departements["surrepresentation"] = (
    (score_departements["score"] - score_departements["score_national"])
    / score_departements["score_national"]
    * 100
).round(2)

score_departements[
    score_departements["code_departement"] == "11"
][["candidat", "score", "score_national", "surrepresentation"]]

### *Question 7 - Visualisation*

In [ ]:
import matplotlib.pyplot as plt

def plot_surrepresentation(score_departements, candidat, top_n):
    # Filtrer candidat
    df_cand = score_departements[
        score_departements["candidat"] == candidat
    ].copy()
    
    # Surreprésentation absolue (intensité)
    df_cand["surrepr_abs"] = df_cand["surrepresentation"].abs()
    
    # Top départements
    df_top = (
        df_cand
        .sort_values("surrepr_abs", ascending=False)
        .head(top_n)
    )
    
    # Tri pour affichage (barres horizontales propres)
    df_top = df_top.sort_values("surrepresentation")
    
    # Plot
    plt.figure()
    plt.barh(df_top["code_departement"], df_top["surrepresentation"])
    
    plt.xlabel("Surreprésentation (%)")
    plt.ylabel("Département")
    plt.title(f"Surreprésentation de {candidat}")
    
    plt.tight_layout()
    plt.show()

plot_surrepresentation(score_departements, "Éric ZEMMOUR", top_n=5)

## **3. Cartographie** 

### *Question 8 - Cartes par candidat*


In [ ]:
!pip install cartiflette

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from cartiflette import carti_download



# Reconstruction de score_departements
df["code_commune"] = df["code_departement"].astype(str).str.zfill(2) + df["code_commune"].astype(str).str.zfill(3)
df["candidat"] = df["prenom"].astype(str) + " " + df["nom"].astype(str)

non_candidats = ["Abstentions", "Blancs", "Nuls", "Exprimés", "Votants", "Inscrits"]
df_expr = df[~df["nom"].isin(non_candidats)].copy()

# Scores nationaux
votes_nationaux = df_expr.groupby("candidat")["voix"].sum().reset_index()
votes_nationaux.columns = ["candidat", "votes_national"]
total_national = votes_nationaux["votes_national"].sum()
votes_nationaux["score_national"] = votes_nationaux["votes_national"] / total_national * 100

# Scores départementaux
votes_dept = df_expr.groupby(["code_departement", "candidat"])["voix"].sum().reset_index()
votes_dept.columns = ["code_departement", "candidat", "votes_departement"]
total_dept = votes_dept.groupby("code_departement")["votes_departement"].sum().reset_index()
total_dept.columns = ["code_departement", "total_dept"]
votes_dept = votes_dept.merge(total_dept, on="code_departement")
votes_dept["score_departement"] = votes_dept["votes_departement"] / votes_dept["total_dept"] * 100

# Fusion national + départemental + surreprésentation
score_departements = votes_dept.merge(votes_nationaux, on="candidat")
score_departements["surrepresentation"] = (
    (score_departements["score_departement"] - score_departements["score_national"])
    / score_departements["score_national"] * 100
)

#  Fond de carte 
departement_borders = carti_download(
    values=["France"],
    crs=4326,
    borders="DEPARTEMENT",
    vectorfile_format="geojson",
    simplification=50,
    filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2022
)

# Fonction cartographie 
def carte_candidat(score_departements, candidat, departement_borders):
    df_cand = score_departements[
        score_departements["candidat"] == candidat
    ].copy()

    df_cand["code_departement"] = df_cand["code_departement"].astype(str).str.zfill(2)
    departement_borders["code_departement"] = (
        departement_borders["INSEE_DEP"].astype(str).str.zfill(2)
    )

    gdf = departement_borders.merge(
        df_cand[["code_departement", "surrepresentation"]],
        on="code_departement",
        how="left"
    )

    vmax = gdf["surrepresentation"].abs().max()
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(8, 7))
    gdf.plot(
        column="surrepresentation",
        cmap="RdBu_r",
        norm=norm,
        linewidth=0.4,
        edgecolor="white",
        ax=ax,
        legend=True,
        legend_kwds={
            "label": "% par rapport\nmoyenne nationale",
            "orientation": "vertical",
            "shrink": 0.6
        }
    )
    ax.set_title(f"Surreprésentation de {candidat}", fontsize=13, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

carte_candidat(score_departements, "Marine LE PEN", departement_borders)
carte_candidat(score_departements, "Emmanuel MACRON", departement_borders)
carte_candidat(score_departements, "Éric ZEMMOUR", departement_borders)